In [ ]:
!pip install sentence-transformers
!pip install matplotlib
!pip install plotly

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"

# Performing Sentence Embeddings and Visualizing it

In [ ]:
import plotly.graph_objects as go
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
import numpy as np

# --- Load SBERT model ---
sbert = SentenceTransformer("all-MiniLM-L6-v2")

# --- Sentences for embedding ---
sentences = [
    "Common symptoms of diabetes include frequent urination, increased thirst, and unexplained weight loss.",
    "High blood pressure often has no noticeable symptoms and is known as the silent killer.",
    "Asthma is a chronic condition that affects the airways and can cause wheezing and shortness of breath.",
    "A persistent cough and chest pain can be symptoms of pneumonia.",
    "Headaches and sensitivity to light are common signs of migraine.",
    "Anemia is a condition where the body does not have enough healthy red blood cells.",
    "Regular exercise can help reduce the risk of heart disease.",
]

# --- Generate embeddings ---
embeddings = sbert.encode(sentences)
embeddings = np.array(embeddings)  # ensure NumPy array

# --- Reduce dimensions to 3D using PCA ---
pca = PCA(n_components=3)
reduced = pca.fit_transform(embeddings)
x, y, z = reduced[:,0], reduced[:,1], reduced[:,2]

# --- Create 3D scatter plot with Plotly ---
fig = go.Figure(data=[go.Scatter3d(
    x=x,
    y=y,
    z=z,
    mode='markers+text',
    text=sentences,
    textposition='top center',
    marker=dict(
        size=8,
        color=z,           # color by z-axis
        colorscale='Viridis',
        opacity=0.8
    )
)])

fig.update_layout(
    scene=dict(
        xaxis_title='PCA 1',
        yaxis_title='PCA 2',
        zaxis_title='PCA 3'
    ),
    title="Sentence Embeddings (SBERT) - 3D PCA projection",
    height=800
)

fig.show()

# Adding a New Sentence and Visualizing it

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Original embeddings (frozen)
original_embeddings = embeddings  # from your first plot

# Fit PCA on original embeddings
pca = PCA(n_components=3)
reduced_original = pca.fit_transform(original_embeddings)

# Embed new sentence
new_sentence = "Fever and body aches are typical symptoms of influenza."
new_embedding = sbert.encode([new_sentence])
new_embedding = np.array(new_embedding)

# Transform new embedding using the same PCA
reduced_new = pca.transform(new_embedding)

# Combine for plotting
reduced_all = np.vstack([reduced_original, reduced_new])
all_sentences = sentences + [new_sentence]

x, y, z = reduced_all[:,0], reduced_all[:,1], reduced_all[:,2]

# --- Step 5: Plot in 3D with Plotly ---
fig = go.Figure(data=[go.Scatter3d(
    x=x,
    y=y,
    z=z,
    mode='markers+text',
    text=all_sentences,
    textposition='top center',
    marker=dict(
        size=8,
        color=z,
        colorscale='Viridis',
        opacity=0.8
    )
)])

fig.update_layout(
    scene=dict(
        xaxis_title='PCA 1',
        yaxis_title='PCA 2',
        zaxis_title='PCA 3'
    ),
    title="Sentence Embeddings (SBERT) - New sentence added",
    height=800
)

fig.show()

# Semantic Matching

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load SBERT
model = SentenceTransformer("all-MiniLM-L6-v2")

# Reference sentences
reference_sentences = [
    "Common symptoms of diabetes include frequent urination, increased thirst, and unexplained weight loss.",
    "High blood pressure often has no noticeable symptoms and is known as the silent killer.",
    "Asthma is a chronic condition that affects the airways and can cause wheezing and shortness of breath.",
    "A persistent cough and chest pain can be symptoms of pneumonia.",
    "Headaches and sensitivity to light are common signs of migraine.",
    "Anemia is a condition where the body does not have enough healthy red blood cells.",
    "Regular exercise can help reduce the risk of heart disease.",
    "Fever and body aches are typical symptoms of influenza."
]

# Embed reference sentences
ref_embeddings = model.encode(reference_sentences)

In [ ]:
while True:    
    query = input("Please input a sentence to match: ")
    query_embedding = model.encode([query])
    
    # Compute cosine similarity
    similarities = cosine_similarity(query_embedding, ref_embeddings)
    similarities = similarities.flatten()  # make 1D array
    
    # Find the best match
    best_idx = np.argmax(similarities)
    best_match = reference_sentences[best_idx]
    best_score = similarities[best_idx]
    
    print(f"Query: {query}")
    print(f"Best Match: {best_match}")
    print(f"Similarity Score: {best_score:.4f}")    

# Questions you can ask

## 🩺 Symptoms-based questions
These test whether the model understands symptoms vs diseases.
- “What are the signs of high blood pressure?”
- “Why do people with pneumonia have chest pain?”
- “What symptoms are associated with migraines?”
- “How does anemia usually present?”
- “What illnesses cause fever and body aches?”

## 🫁 Condition understanding
These focus on what a disease is, not just symptoms.
- “What is asthma?”
- “Explain what anemia means”
- “What kind of condition is influenza?”
- “Is high blood pressure a chronic illness?”
- “What happens in the body during heart disease?”

## ❤️ Prevention & lifestyle
These test semantic links to risk reduction and prevention.
- “How can I reduce my risk of heart disease?”
- “Does exercise help prevent cardiovascular problems?”
- “What lifestyle changes improve heart health?”
- “Can physical activity lower health risks?”

## 🔍 Vague or short queries (great for demos)
These show why embeddings beat keyword search.
- “diabetes symptoms”
- “silent killer disease”
- “chronic airway condition”
- “low red blood cells”
- “flu symptoms”

## 🧠 Harder paraphrases (advanced demo)
These are excellent for showing true semantic understanding.
- “A patient has frequent urination and extreme thirst — what could this indicate?”
- “Which condition often shows no symptoms but increases stroke risk?”
- “What illness causes wheezing due to airway inflammation?”
- “What condition leads to insufficient oxygen transport in the blood?”
- “What disease causes headaches and sensitivity to light?” 